In [ ]:
import sys

sys.path.append('/Users/wjs/work/pyproj/SlideGen/slidegen')
sys.path.append('/Users/wjs/work/pyproj/SlideGen')

# from slidegen.core.docparse.parsers import DocxParser
# from slidegen.core.docparse.markdown_document import Element, Heading, CodeBlock, Paragraph, Table
# from slidegen.core.presentation.pptpages import ChapterHomePage, CoverPage
from pptx.enum.shapes import MSO_SHAPE_TYPE

from slidegen.workflows.docparse import DocumentReader, MarkdownDocument

In [ ]:
converter = DocumentReader()
file_content = converter.convert("/Users/wjs/work/pyproj/SlideGen/test/data/Snipaste.docx")
file_content.text_content

In [2]:
# 读取Markdown文件内容
with open("data/test.md") as f:
    md_content = f.read()
doc = MarkdownDocument(md_content)

In [5]:
doc.main.element_text_source

'# 深度学习原理与架构'

In [6]:
print(doc.main.contents[2])
print(doc.main.contents[2].contents)
print("--------------------------------")
print(doc.main.contents[2].contents[0])
print(doc.main.contents[2].contents[0].text)

<Heading level=2 text=1. 深度学习概述>
[<Heading level=3 text=1.1 什么是深度学习？>, <Heading level=3 text=1.2 深度学习的历史>, <Heading level=3 text=1.3 深度学习的应用领域>]
--------------------------------
<Heading level=3 text=1.1 什么是深度学习？>
深度学习是机器学习的一个子领域，专注于使用多层神经网络来模拟复杂的数据模式。
通过多层次的非线性变换，深度学习能够自动提取特征并进行高级抽象。


In [12]:
doc.main.contents[0]

<Paragraph text='---'>

In [60]:
children = doc.contents[0].contents
children[2].contents[0].contents

[<Paragraph text='深度学习是机器学习的一个子领域，专注于使用多层神经网络来模拟复杂的数据模式。'>,
 <Paragraph text='通过多层次的非线性变换，深度学习能够自动提取特征并进行高级抽象。'>]

In [4]:
import pptx

# path = "/Users/wjs/work/pyproj/SlideGen/test/data/竞选学生会主席.pptx"
path = "/Users/wjs/work/pyproj/SlideGen/test/data/DeepSeek对中国AI产业的影响.pptx"
# path = "/Users/wjs/work/pyproj/SlideGen/test/data/深度学习原理架构与应用.pptx"
# path = "/Users/wjs/work/pyproj/SlideGen/test/data/紫藤萝瀑布教学课件.pptx"
# path = "/Users/wjs/work/pyproj/SlideGen/test/data/创意复古民国风PPT模板.pptx"
# path = "/Users/wjs/work/pyproj/SlideGen/test/data/清新简约手绘桌面通用PPT模板.pptx"
path = "/Users/wjs/work/pyproj/SlideGen/test/test_ppt.pptx"
presentation = pptx.Presentation(path)
slide_index = 1


In [5]:
import json

import pptx
import pptx.presentation
from lxml import etree
from pptx.oxml import parse_xml

slide = presentation.slides[slide_index]


In [38]:
ChapterHomePage.selected_style = 1



In [40]:
print(ChapterHomePage.selected_style)
cp = ChapterHomePage()
print(cp.selected_style)

1
1


In [6]:
def remove_custDataLst(xml_str: str) -> str:
    """
    Remove the <p:custDataLst> part in the XML and return the processed XML string.
    
    Args:
        xml_str (str): The input XML string, containing the <p:custDataLst> part.
        
    Returns:
        str: The processed XML string, without the <p:custDataLst> part.
    """
    root = etree.fromstring(xml_str)
    ns = root.nsmap

    cust_data_list = root.find(".//p:custDataLst", namespaces=ns)

    if cust_data_list is not None:
        parent = cust_data_list.getparent()
        if parent is not None:
            parent.remove(cust_data_list)

    return etree.tostring(
        root,
        encoding="unicode",
        pretty_print=True,
        xml_declaration=False,
    )


In [8]:
# slide = presentation.slides[7]
def get_slide_shapes_data(slide):
    shapes_data = [] # 初始化一个列表来存储所有形状的信息
    for shape in slide.shapes:
        shape_info = {
            "shape_name": shape.name,
            "shape_type": str(shape.shape_type) # 将枚举类型转换为字符串以便JSON序列化
        }
        if shape.has_text_frame:  # 检查 shape 是否有文本框
            text = shape.text.strip()
            if shape.is_placeholder:  # 处理没有文本的占位符
                placeholder_idx = shape.placeholder_format.idx  # 获取占位符索引
                text = shape.text  # 使用形状名称生成占位符文本
                print(f"Shape Type: {shape.placeholder_format.type}, \
                    Shape Text: {text}, Shape Name: {shape.name}")
                shape_info["placeholder_type"] = str(shape.placeholder_format.type) # 转换枚举类型
                shape_info["shape_text"] = text
                # 如果需要，也可以添加占位符索引
                # shape_info["placeholder_idx"] = placeholder_idx
            else:
                print(f"Shape Type: {shape.shape_type}, Shape Text: {text}, Shape Name: {shape.name}")
                # print(shape.element.xml)
                shape_info["shape_text"] = text
                shape_info["xml"] = remove_custDataLst(shape.element.xml)
        elif shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
            print(f"Shape Type: {shape.shape_type}, Shape Name: {shape.name}")
            print(shape.element.xml)
            # 保存图片
            # with open(f"/Users/wjs/work/pyproj/SlideGen/components/picture/{shape.name}.png", "wb") as f:
            #     f.write(shape.image.blob)
            shape_info["xml"] = remove_custDataLst(shape.element.xml)

        else:
            print(f"Shape Type: {shape.shape_type}, Shape Name: {shape.name}")
            # print(shape.element.xml)
            shape_info["xml"] = remove_custDataLst(shape.element.xml)
        shape_info["x"] = shape.left
        shape_info["y"] = shape.top
        shape_info["width"] = shape.width
        shape_info["height"] = shape.height
        shapes_data.append(shape_info)
    return shapes_data

output_json_path = 'slide_shapes_9.json'

shapes_data = get_slide_shapes_data(slide)
print(shapes_data)

try:
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(shapes_data, f, ensure_ascii=False, indent=4)
    print(f"形状数据已成功保存到 {output_json_path}") # 打印成功消息
except Exception as e:
    print(f"保存JSON文件时出错: {e}") # 打印错误信息

Shape Type: TITLE (1),                     Shape Text: 目录, Shape Name: 标题
Shape Type: AUTO_SHAPE (1), Shape Text: , Shape Name: 圆角矩形 27
Shape Type: AUTO_SHAPE (1), Shape Text: 01, Shape Name: 圆角矩形 34
Shape Type: TEXT_BOX (17), Shape Text: -、产品概述, Shape Name: 项标题
Shape Type: AUTO_SHAPE (1), Shape Text: , Shape Name: 圆角矩形 27
Shape Type: AUTO_SHAPE (1), Shape Text: 02, Shape Name: 圆角矩形 34
Shape Type: TEXT_BOX (17), Shape Text: 二、市场分析, Shape Name: 项标题
Shape Type: AUTO_SHAPE (1), Shape Text: , Shape Name: 圆角矩形 27
Shape Type: AUTO_SHAPE (1), Shape Text: 03, Shape Name: 圆角矩形 34
Shape Type: TEXT_BOX (17), Shape Text: 三、盈利模式, Shape Name: 项标题
Shape Type: AUTO_SHAPE (1), Shape Text: , Shape Name: 圆角矩形 27
Shape Type: AUTO_SHAPE (1), Shape Text: 04, Shape Name: 圆角矩形 34
Shape Type: TEXT_BOX (17), Shape Text: 四、产品特色, Shape Name: 项标题
Shape Type: AUTO_SHAPE (1), Shape Text: , Shape Name: 圆角矩形 27
Shape Type: AUTO_SHAPE (1), Shape Text: 05, Shape Name: 圆角矩形 34
Shape Type: TEXT_BOX (17), Shape Text: 五、未来发

In [37]:
new_slide = presentation.slides.add_slide(slide.slide_layout)
# new_slide.placeholders[0].text = "Hello World!"
presentation.save("test____.pptx")

In [16]:
print(get_slide_shapes_data(new_slide))

Shape Type: BODY (2),                     Shape Text: [Placeholder Text for Text Placeholder 1], Shape Name: Text Placeholder 1, place
Shape Type: TITLE (1),                     Shape Text: [Placeholder Text for Title 2], Shape Name: Title 2, place
[{'shape_name': 'Text Placeholder 1', 'shape_type': 'PLACEHOLDER (14)', 'placeholder_type': 'BODY (2)', 'shape_text': '[Placeholder Text for Text Placeholder 1]', 'x': 9605355, 'y': 4452317, 'width': 1540800, 'height': 424800}, {'shape_name': 'Title 2', 'shape_type': 'PLACEHOLDER (14)', 'placeholder_type': 'TITLE (1)', 'shape_text': '[Placeholder Text for Title 2]', 'x': 6096000, 'y': 1985038, 'width': 5079968, 'height': 1869371}]


In [27]:

picture_shape = new_slide.shapes.add_picture("/Users/wjs/work/pyproj/SlideGen/components/picture/transparent/ write.png", 3202623, 1696085, width=323642, height=323642)


In [30]:
picture_shape.image

'a6d15e036866681b9f4aa8e26343b56b721792a6'

In [11]:
xml_ = """<p:sp xmlns:p=\"http://schemas.openxmlformats.org/presentationml/2006/main\" xmlns:a=\"http://schemas.openxmlformats.org/drawingml/2006/main\" xmlns:r=\"http://schemas.openxmlformats.org/officeDocument/2006/relationships\">\n  <p:nvSpPr>\n    <p:cNvPr id=\"60\" name=\"项标题\"/>\n    <p:cNvSpPr txBox=\"1\"/>\n    <p:nvPr>\n      </p:nvPr>\n  </p:nvSpPr>\n  <p:spPr>\n    <a:xfrm>\n      <a:off x=\"4344158\" y=\"4081002\"/>\n      <a:ext cx=\"1628618\" cy=\"1000125\"/>\n    </a:xfrm>\n    <a:prstGeom prst=\"rect\">\n      <a:avLst/>\n    </a:prstGeom>\n    <a:noFill/>\n  </p:spPr>\n  <p:txBody>\n    <a:bodyPr wrap=\"square\" lIns=\"0\" tIns=\"0\" rIns=\"0\" bIns=\"0\" rtlCol=\"0\" anchor=\"ctr\">\n      <a:noAutofit/>\n    </a:bodyPr>\n    <a:lstStyle/>\n    <a:p>\n      <a:pPr algn=\"ctr\">\n        <a:lnSpc>\n          <a:spcPct val=\"100000\"/>\n        </a:lnSpc>\n      </a:pPr>\n      <a:r>\n        <a:rPr lang=\"zh-CN\" altLang=\"en-US\" b=\"1\" spc=\"300\" dirty=\"0\">\n          <a:solidFill>\n            <a:srgbClr val=\"FFFFFF\"/>\n          </a:solidFill>\n          <a:latin typeface=\"+mn-ea\"/>\n        </a:rPr>\n        <a:t>三、盈利模式</a:t>\n      </a:r>\n    </a:p>\n    <a:p/>\n  </p:txBody>\n</p:sp>\n"""

new_shape = slide.shapes._shape_factory(
    slide.shapes._spTree.insert_element_before(parse_xml(xml_), 'p:extLst'))


In [17]:
len(new_shape.text_frame.paragraphs)

2

In [12]:
new_shape.text

'三、盈利模式\n'

In [60]:
presentation.save("test____.pptx")

In [1]:
from enum import IntEnum


class ChapterLayout(IntEnum):
    """Chapter layout type"""

    str_value: str

    ONE_POINT = (1, "one_point")
    TWO_POINTS = (2, "two_points")
    THREE_POINTS = (3, "three_points")
    FOUR_POINTS = (4, "four_points")

    def __new__(cls, number: int, str_value: str) -> "ChapterLayout":
        obj = int.__new__(cls, number)
        obj._value_ = number
        obj.str_value = str_value
        return obj

    def __str__(self) -> str:
        return f"{self.name} ({self.value})"


In [8]:
str(ChapterLayout(1))

'ONE_POINT (1)'

In [2]:
from openai import OpenAI

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="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",
)

completion = client.chat.completions.create(
  extra_body={},
  model="deepseek/deepseek-chat-v3-0324:free",
  messages=[
    {
      "role": "user",
      "content": "What is the meaning of life?"
    }
  ]
)
print(completion.choices[0].message.content)

The meaning of life is one of the most profound and debated questions in philosophy, religion, science, and personal reflection. Different perspectives offer various answers:

### 1. **Philosophical Perspectives**  
   - **Existentialism (e.g., Sartre, Camus):** Life has no inherent meaning—we must create our own purpose through choices, actions, and passions.  
   - **Absurdism (Camus):** The search for meaning in a meaningless universe is absurd, but we must embrace life regardless (e.g., "The Myth of Sisyphus").  
   - **Stoicism:** Meaning comes from virtue, resilience, and aligning with nature or reason.  

### 2. **Religious/Spiritual Views**  
   - **Theistic religions (Christianity, Islam, etc.):** Life’s purpose is to serve, love, or unite with a divine being or follow a sacred path.  
   - **Buddhism/Hinduism:** Meaning lies in enlightenment (nirvana) or liberation (moksha) from suffering and the cycle of rebirth.  
   - **Pantheism:** Meaning is found in unity with the unive

In [3]:
from enum import Enum

class ContentType(str, Enum):
    """Enum for content types supported by knowledge readers."""

    # Generic types
    FILE = "file"
    URL = "url"
    TEXT = "text"
    TOPIC = "topic"
    YOUTUBE = "youtube"

    # Document file extensions
    PDF = ".pdf"
    TXT = ".txt"
    MARKDOWN = ".md"
    DOCX = ".docx"
    DOC = ".doc"
    JSON = ".json"
    HTML = ".html"
    HTM = ".htm"

    # Spreadsheet file extensions
    CSV = ".csv"
    XLSX = ".xlsx"
    XLS = ".xls"



In [8]:
ext = ".html"
if ext in [ContentType.HTML.value, ContentType.HTM.value]:
    print("html")
else:
    print("not html")



html


In [7]:
from agno.knowledge.embedder.openai import OpenAIEmbedder

embedder = OpenAIEmbedder(
    api_key="dummy",
    base_url="http://192.168.1.144:8000/v1",
    id="null",
)

embedder.get_embedding("Hello, world!")

[-0.010912647,
 0.006483564,
 -0.07969505,
 0.0052271574,
 0.026294466,
 0.06406905,
 -0.03370945,
 0.012813046,
 0.019233169,
 -0.03808565,
 -0.0045172526,
 0.03214357,
 0.037252877,
 0.012112128,
 -0.051215265,
 0.051676422,
 0.06712766,
 0.0067708828,
 0.010241504,
 -0.032361966,
 0.033543393,
 0.0142532615,
 -0.023349112,
 0.025777798,
 0.033944838,
 -0.008884619,
 -0.011483693,
 0.0055084378,
 0.02630225,
 -0.014898526,
 -0.021315966,
 -0.0033294735,
 -0.031588268,
 -0.0045092604,
 -0.010690417,
 0.018577842,
 -0.0024147218,
 0.029853337,
 0.026953943,
 0.037011176,
 -0.0042833,
 -0.022595959,
 0.0039431844,
 -0.04616072,
 0.028629135,
 0.0336683,
 0.02835596,
 0.052278437,
 0.012797813,
 0.0040558963,
 0.0026350708,
 -0.02007563,
 0.0136256,
 0.027194317,
 0.0037977304,
 0.030488454,
 0.059764143,
 0.0016813042,
 -0.0066122934,
 -0.08206406,
 -0.019528983,
 0.062525176,
 0.035793312,
 0.020712543,
 -0.0018282295,
 -0.008140493,
 0.015553557,
 -0.017369842,
 0.018137146,
 -0.04848